# Platform & Workspace

You get to know the Databricks Lakehouse platform: its architecture, the workspace components, the basics of Unity Catalog and the compute types. You also learn the principles of cost management and how to work in the Databricks GUI: notebooks, markdown, SQL and Python in one notebook, simple visualizations and Genie Code (formerly Databricks Assistant), the built-in AI assistant.

## Learning Objectives

After completing this module you will be able to:

- Describe the Databricks Lakehouse architecture and how it combines the benefits of a data lake and a data warehouse
- Navigate the Databricks workspace: notebooks, clusters, DBFS, Catalog Explorer, Git Folders
- Use the GUI well: write markdown, mix SQL and Python in the same notebook
- Create simple visualizations directly in the notebook GUI
- Understand how Genie Code supports daily work in notebooks
- Describe the 3-level namespace of Unity Catalog: `catalog.schema.table`
- Explain the difference between All-Purpose clusters and Job clusters and what each one costs
- Understand autoscaling and how it affects costs
- Apply basic cost management: the right cluster type, sizing, auto-termination


## Databricks Platform Elements

The main elements of the Databricks platform are the Workspace, Catalog Explorer, Git Folders and Volumes. A data engineer works with all of them every day, so you need to know what each one is for.

### Per-user Isolation

Run the setup notebook first. It gives you your own catalog and schemas, so your tables do not collide with the tables of other participants.


In [0]:
%run ../../setup/00_setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# 00_setup above printed your catalog, schemas and dataset path: CATALOG, BRONZE_SCHEMA, ... and DATASET_PATH

The output shows the catalog, schemas and dataset path that the rest of this notebook uses.

### Comparison of Traditional Architecture vs Lakehouse

Compare the traditional approach (a data lake plus a data warehouse) with the Lakehouse.

![Evolution of data storage: data warehouse, data lake and lakehouse, with the advantages and disadvantages of each](../../../assets/images/4c9090bd82f2475c810bafde13f978e0.png)

Before and now: separate tools vs one platform:

<img src="../../../assets/images/nb01-many-tools-to-one-platform.png" alt="From many tools to one platform: before, separate storage, data lake, ETL, data warehouse, BI and AI/ML tools, each with its own security, joined by integration tools, connectors and sync jobs that copy data; now, one Data Intelligence Platform where all workloads use one copy of data in open formats under Unity Catalog governance" width="800">

Lakehouse benefits:
- Single copy of data (single source of truth)
- Lower storage costs
- No synchronization latency
- Common governance for all use cases

### Key Components

The Databricks platform has several components. Together they give you a complete environment for working with data in the Lakehouse architecture.

| Component | What it is |
|---|---|
| Workspace | Working environment with notebooks, experiments, folders and resources |
| Catalog Explorer | Interface for managing catalogs, schemas, tables and views |
| Git Folders (formerly Repos) | Git integration for versioning notebooks and code |
| Volumes | Storage for unstructured files (images, models, artifacts) |
| DBFS (Databricks File System), legacy: use Volumes instead | Virtual file system over cloud storage |

In practice:
- The Workspace organizes projects and team collaboration.
- Catalog Explorer is where you explore data and manage governance.
- Git Folders connect your development workflow with Git.

### Workspace Exploration

#### Example: Workspace Exploration

Find each part of the workspace screen in your own workspace.

Workspace elements:
1. Sidebar (left side):
   - Workspace: folders, notebooks and Git folders (create one with **Create** → **Git folder** in the Workspace browser)
   - Compute: cluster management
   - Jobs & Pipelines: Lakeflow Jobs (formerly Workflows) and pipelines
   - Catalog: Unity Catalog explorer

2. Main panel: notebook editor or details view

3. Notebook toolbar: the compute selector (serverless or a cluster) and the run buttons. The top bar holds search, help and your account.

Use the left menu to switch between sections. In the Catalog section you browse catalogs, schemas and tables. In the Compute section you manage Spark clusters.

#### Databricks Workspace UI

<img src="../../../assets/images/ss-01-workspace.png" width="800">

The screenshot was taken on an earlier version of this notebook. The screen layout is the same.

### Catalog Explorer: Unity Catalog Structure

#### Example: Catalog Explorer

Unity Catalog keeps every object in a hierarchy. Catalog Explorer (**Catalog** in the sidebar) shows this hierarchy.

#### Catalog Explorer Screenshot

<img src="../../../assets/images/ss-01-catalog.png" width="800">

The screenshot shows the trainer's catalog. In your workspace you see your own catalog, `retailhub_<your_name>`.

A table name without a catalog and schema points to the current catalog and schema. Check them before you write such short names in your code.


In [0]:
# Short table names resolve against these two
display(spark.sql("SELECT current_catalog() AS current_catalog, current_schema() AS current_schema"))

The output shows the catalog and schema that a short table name points to.

Unity Catalog hierarchy:

<img src="../../../assets/images/nb01-uc-namespace.png" alt="Unity Catalog hierarchy metastore, catalog, schema, table and volume, with a three-level name split into catalog, schema and table" width="800">

> 3-level namespace: `catalog.schema.object`, for example `prod.gold.sales_summary`.

> Access follows the hierarchy. To read a table you need `USE CATALOG` on its catalog, `USE SCHEMA` on its schema and `SELECT` on the table. `SELECT` alone is not enough. Module 10 (`10_governance_and_security`) covers privileges.

### Browsing Catalogs and Schemas

#### Example: Browsing Catalogs and Schemas

You can list Unity Catalog objects with SQL commands, not only in Catalog Explorer. You need this in scripts and jobs, where there is no UI.

In [0]:
# Catalogs of this course; drop LIKE to list every catalog you can use
display(spark.sql("SHOW CATALOGS LIKE 'retailhub*'"))

In [0]:
# List the schemas in your course catalog
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

`SHOW SCHEMAS` returns one row for each schema in your catalog.

### Git Folders

In production, notebooks live in **Git Folders** (Repos), under version control in GitHub, GitLab or Azure DevOps.

The usual workflow is feature branch → pull request → merge → bundle deploy with Declarative Automation Bundles (formerly Databricks Asset Bundles).

Module `09_cicd_and_bundles` covers this in detail.


### Volumes vs DBFS

A Volume holds files that are not tables: raw CSV and JSON drops, images, model artifacts. It lives in a schema next to the tables, so Unity Catalog governs it (`READ VOLUME` / `WRITE VOLUME` privileges, audit, lineage). Its path is `/Volumes/<catalog>/<schema>/<volume>/...`. The same path works in `dbutils.fs`, `spark.read`, `read_files()` and SQL `LIST`.

- `dbfs:/` (DBFS root, mounts) is a **legacy** layer of older workspaces. New Unity Catalog workspaces use Volumes.
- A managed Volume (no `LOCATION`) stores its files in the managed storage of the schema.
- An external Volume (`CREATE EXTERNAL VOLUME ... LOCATION 'abfss://...'`) points at an existing cloud path.

The next four cells create a managed Volume, write a file, list it and clean up. Create a Volume whenever raw files need a place that Unity Catalog governs.

In [0]:
# Create a managed Volume in your bronze schema (IF NOT EXISTS: safe on a second run)
DEMO_VOLUME = f"{CATALOG}.{BRONZE_SCHEMA}.m01_demo_files"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {DEMO_VOLUME} COMMENT 'Module 01 demo files'")
display(spark.sql(f"DESCRIBE VOLUME {DEMO_VOLUME}"))

Look at `volume_type` = `MANAGED` and `storage_location`. Unity Catalog chose the location because you did not give one.

Write a small text file with `dbutils.fs.put` and read it back with `dbutils.fs.head`. Use this pattern for config files or for a quick check that you can write to a path.

In [0]:
# Write a text file into the Volume and read it back
DEMO_DIR = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/m01_demo_files/notes"
NOTE_TEXT = "RetailHub raw files land here."
dbutils.fs.put(f"{DEMO_DIR}/readme.txt", NOTE_TEXT, overwrite=True)  # overwrite: rerun-safe
note = dbutils.fs.head(f"{DEMO_DIR}/readme.txt")
print(note)

`dbutils.fs.put` creates the missing `notes/` folder. `head` returns the first bytes of the file as a string (64 KB by default).

List the folder twice: with `dbutils.fs.ls` in Python and with `LIST` in SQL. Both return one row per file with its path, name and size. `dbutils.fs.ls` writes the path as `dbfs:/Volumes/...` and `LIST` as `/Volumes/...`: it is the same file in the volume. `modificationTime` is in milliseconds since 1970 (Unix time).

In [0]:
# List the folder from Python and from SQL
display(dbutils.fs.ls(DEMO_DIR))
display(spark.sql(f"LIST '{DEMO_DIR}'"))

In [0]:
# Clean up: remove the demo folder and the Volume
dbutils.fs.rm(DEMO_DIR, recurse=True)
spark.sql(f"DROP VOLUME IF EXISTS {DEMO_VOLUME}").collect()
print(f"Removed {DEMO_DIR} and dropped the volume {DEMO_VOLUME}")

## Magic Commands

A magic command on the first line of a cell switches the language of the cell or runs a system operation.

| Command | Purpose |
|---------|---------|
| `%sql` | SQL cell |
| `%python` | Python cell (default) |
| `%md` | Markdown documentation |
| `%fs` | DBFS file operations |
| `%sh` | Shell commands |
| `%run` | Execute another notebook |
| `%pip` | Install notebook-scoped libraries |

### Commenting / Uncommenting Lines

| Action | Mac | Windows / Linux |
|--------|-----|-----------------|
| Toggle comment on selected lines | `Cmd + /` | `Ctrl + /` |


### The %sql magic command

Use `%sql` when a cell is pure SQL. You do not need a Python wrapper, and the result shows as a table.

In [0]:
%sql
-- %sql runs the whole cell as SQL, no spark.sql() wrapper needed

SELECT 
  current_catalog() as catalog,
  current_schema() as schema,
  current_user() as user,
  current_timestamp() as timestamp

### Mixing Python and SQL

A temp view connects the two languages. Create data in Python, register it as a temp view and query it with SQL. Use this when one part of the logic is easier in Python and the rest is easier in SQL.

In [0]:
# Sample rows defined in Python
data = [
    (1, "Alice", "Engineering", 95000),
    (2, "Bob", "Sales", 75000),
    (3, "Charlie", "Engineering", 105000),
    (4, "Diana", "Marketing", 68000),
    (5, "Eve", "Engineering", 98000)
]

# Explicit schema, so Spark does not have to infer the types
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False)
])

In [0]:
# Build a DataFrame from the rows and the schema
df = spark.createDataFrame(data, schema)
display(df)

In [0]:
# The temp view makes the DataFrame visible to SQL cells in this session
df.createOrReplaceTempView("employees_temp")

In [0]:
%sql
-- Aggregate the Python data with SQL through the temp view

SELECT 
  department,
  COUNT(*) as employee_count,
  ROUND(AVG(salary), 0) as avg_salary,
  MAX(salary) as max_salary
FROM employees_temp
GROUP BY department
ORDER BY avg_salary DESC

The result has one row per department, sorted by average salary.

### Genie Code (AI assistant)

> **Self-study:** skipped in class, not required by any lab.

Databricks has a built-in AI assistant, **Genie Code** (formerly Databricks Assistant). It knows the context of your data, for example the table schemas in Unity Catalog.

How to open it:
1. Press **Cmd+I** (Mac) or **Ctrl+I** (Windows) inside a cell.
2. Open the Genie Code side panel.

What it is for:

| Task | Example |
|---|---|
| Code generation | "Write a SQL query that calculates average sales by region from the sales table". |
| Code explanation | Select a complex snippet and ask "Explain this code". |
| Fixing errors | When a cell returns an error, click "Diagnose Error". The assistant explains the cause and proposes a fix. |
| Transformation | "Rewrite this code from PySpark to SQL". |


## Unity Catalog

Unity Catalog is a modern metadata management system that replaces the Hive Metastore. It gives you central access control, lineage and governance across the whole organization.

### Theoretical Introduction

Databricks supports two metadata systems: the legacy Hive Metastore and the modern Unity Catalog. Use Unity Catalog for all new projects, because it has the advanced governance and security features.

Key differences:

| Aspect | Hive Metastore | Unity Catalog |
|--------|----------------|---------------|
| Governance | Limited | Full: RBAC, masking, audit |
| Namespace | 2-level (db.table) | 3-level (catalog.schema.table) |
| Cross-workspace | No | Yes (shared metastore) |
| Lineage | None | End-to-end lineage |
| Data Sharing | Limited | Delta Sharing protocol |
| Isolation | Workspace-level | Catalog-level |

Why Unity Catalog:
- Central access management for all workspaces
- Automatic lineage for audit and compliance
- Fine-grained permissions (column-level, row-level)
- Integration with external systems (Delta Sharing)

### Creating a Table in Unity Catalog

#### Example: Creating a Table

A table name in Unity Catalog has three parts: catalog, schema and table. Use the full name in your code, so it does not depend on the current catalog and schema.

In [0]:
# Table name in Unity Catalog (3-level namespace)
table_name = f"{CATALOG}.{BRONZE_SCHEMA}.retail_sales"

# Sales dataset: 5 regions × 4 categories, enough variety for charts
sales_data = [
    ("North",   "Electronics",  142500, 1180, 120.76),
    ("North",   "Apparel",       58300,  940,  62.02),
    ("North",   "Food",          34200,  870,  39.31),
    ("North",   "Home",          71400,  620, 115.16),
    ("South",   "Electronics",   98700,  820, 120.37),
    ("South",   "Apparel",       45100,  730,  61.78),
    ("South",   "Food",          61800, 1540,  40.13),
    ("South",   "Home",          53200,  470, 113.19),
    ("East",    "Electronics",  175300, 1450, 120.90),
    ("East",    "Apparel",       67900, 1090,  62.29),
    ("East",    "Food",          48500, 1210,  40.08),
    ("East",    "Home",          89100,  780, 114.23),
    ("West",    "Electronics",  210400, 1740, 120.92),
    ("West",    "Apparel",       82600, 1330,  62.11),
    ("West",    "Food",          55900, 1380,  40.51),
    ("West",    "Home",         103700,  910, 113.96),
    ("Central", "Electronics",   76300,  630, 121.11),
    ("Central", "Apparel",       31400,  510,  61.57),
    ("Central", "Food",          28700,  710,  40.42),
    ("Central", "Home",          42800,  380, 112.63),
]

In [0]:
sales_schema = StructType([
    StructField("region",          StringType(),  False),
    StructField("category",        StringType(),  False),
    StructField("revenue",         IntegerType(), False),
    StructField("transactions",    IntegerType(), False),
    StructField("avg_order_value", DoubleType(),  False),
])

sales_df = spark.createDataFrame(sales_data, sales_schema)
display(sales_df)

In [0]:
# Save as a Delta table in Unity Catalog under its 3-level name
sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"Table saved: {table_name}")

In [0]:
# Read the table back to check the write
display(spark.table(table_name))

The table has 20 rows: 5 regions × 4 categories.

#### Quick chart in the notebook

In the result of the cell above, click **`+`** → **Visualization**. Choose **Bar**, set the X column to `region` and the Y column to `revenue` (Sum), group by `category`, then click **Save**. The chart is saved with the cell output.

### Summary Statistics

Before an analysis, look at count, mean, min, max and quartiles of every column. `summary()` computes them on any compute. It returns every value as a string, so numbers keep all their decimals and `mean` is `null` for text columns. On classic compute the result widget also offers **`+`** → **Data Profile**; on serverless that option is greyed out ("Data profiles are not supported on serverless compute").

> Use it before an analysis to spot nulls, outliers and unexpected distributions quickly.

In [0]:
# Statistics for every column; works on serverless, unlike the Data Profile widget
display(spark.table(table_name).summary())

### Comparison: PySpark vs SQL

> **Self-study:** skipped in class, not required by any lab. Side-by-side reference: `docs/ENG/pyspark_vs_sparksql.pdf`.

The next cells run the same query twice: the West region, four columns, sorted by revenue. First the DataFrame API (PySpark):

In [0]:
# PySpark: the DataFrame API
df_pyspark = spark.table(table_name)

In [0]:
result_pyspark = df_pyspark \
    .filter(F.col("region") == "West") \
    .select("region", "category", "revenue", "transactions") \
    .orderBy(F.desc("revenue"))
display(result_pyspark)

The same query in SQL:

In [0]:
df = spark.sql(f"""
    SELECT region, category, revenue, transactions
    FROM {table_name}
    WHERE region = 'West'
    ORDER BY revenue DESC
""")
display(df)

Both outputs have the same rows in the same order.

Comparison:
- Performance is identical. Both queries (same columns, same filter, same ORDER BY) compile to the same Catalyst query plan.
- Use PySpark for:
  - complex business logic with UDFs
  - dynamic pipelines (parameterization, loops)
  - integration with Python libraries (pandas, scikit-learn)
- Use SQL for:
  - simple transformations and aggregations
  - a team with strong SQL skills
  - migration from a traditional data warehouse
  - better support for business analysts

## Compute Resources

The compute you pick decides what your code can run and how you pay for it. In this course everything runs on serverless. You still need to know the other types for the exam and for workloads that serverless does not support.

### Compute Options Comparison

| Type | Startup | Billing | Limits | Use for |
|------|---------|---------|--------|---------|
| Serverless (notebooks, jobs, pipelines) | Jobs and pipelines: typically under a minute (performance-optimized, default) or 4–6 min (standard mode). Notebooks: no cluster to configure | DBUs for the compute used; the VM cost is included in the DBU rate | See the list in the next cell (no R, no RDD, no Spark UI, …) | Default for notebooks, jobs and pipelines, used in all labs |
| All-purpose compute (classic) | Minutes (VMs start in your Azure subscription) | DBUs + VMs for the whole uptime, including idle time until auto-termination | You choose size, runtime and access mode (next cells) | Interactive work serverless does not support: R, RDD API, ML runtime on GPU, Spark UI |
| Job compute (classic) | Minutes, for each job run | DBUs (lower jobs rate) + VMs only while the run lasts | Created and terminated by the job | Scheduled production jobs that need classic features |
| SQL warehouse | Serverless: seconds. Pro / Classic: minutes | DBUs while the warehouse runs; auto-stop when idle | SQL only: Python cells do not run | SQL editor, AI/BI dashboards, Genie, BI tools |
| AI Runtime (serverless GPU, Public Preview) | Managed by Databricks | DBUs for the GPU compute used | Deep learning frameworks, A10 / H100 GPUs | Training, fine-tuning and batch inference of deep learning models |

### Cost Optimization Strategies

1. Right-size clusters.
   - Development: 2-4 workers, smallest instance type
   - Production: autoscaling 2-10 workers, based on the workload
2. Use spot (preemptible) instances.
   - 60-80% cost savings for workers
   - Keep the driver on an on-demand instance for stability
   - Trade-off: a job may be interrupted
3. Use the Photon engine.
   - 2-3x faster for aggregations and joins
   - About 2x the DBU cost, but it finishes faster, so it is often cheaper
   - Enable it for large scans, aggregations and joins
   - Skip it for simple transformations and ML training
4. Use cluster policies.
   - Enforce a maximum worker count
   - Require autoscaling
   - Set auto-termination (for example 30 min idle)

### Compute Overview: Serverless or Classic

<img src="../../../assets/images/nb01-compute-choice.png" alt="Which compute runs my code: serverless (default for notebooks, jobs and pipelines), all-purpose and job compute (classic), and SQL warehouses, with their limits" width="800">



### Serverless Compute: the new default in Databricks

> Serverless notebooks and jobs are GA on Azure Databricks and available by default in Unity Catalog workspaces in supported regions.

What is serverless?
- Databricks manages the infrastructure, so **you do not configure a cluster**.
- You pay only for the resources you actually use (seconds of execution, not cluster uptime).
- Resources scale automatically in the background, and you do not see it.

Serverless models in Databricks:

| Context | Serverless compute |
|----------|-------------------|
| Notebooks (serverless) | No cluster to start or configure; the session ends when you detach or after idle time |
| Lakeflow Jobs (serverless jobs) | Each run gets its own isolated compute |
| SQL Warehouse (Serverless) | Fastest type, dedicated to SQL / BI |
| Lakeflow Spark Declarative Pipelines (formerly DLT), serverless | Pipelines without managing compute |

Limits of serverless you will meet in this course (full list: [Serverless compute limitations](https://learn.microsoft.com/azure/databricks/compute/serverless/limitations)):
- No R and no RDD API: only Spark Connect APIs.
- Only a few `spark.conf` keys can be set (for example `spark.sql.shuffle.partitions`). Others raise `CONFIG_NOT_AVAILABLE`.
- No Spark UI: use the **query profile** instead.
- No global temp views (use session temp views) and no `CACHE TABLE`.
- Streaming triggers: only `AvailableNow` and `Once`, no `processingTime`.
- ANSI mode is on: an invalid cast raises an error instead of returning `NULL`.
- No compute-scoped init scripts or libraries: use the notebook **Environment** panel.

For a GPU, use **AI Runtime** (serverless GPU, Public Preview) or a classic cluster with the ML runtime in **Dedicated** access mode.

> To run a notebook on serverless, select "Connect to compute" → "Serverless" in the top-right corner.


### SQL Warehouse

A SQL warehouse is a SQL engine optimized for BI and ad-hoc analytics. It is an alternative to notebook clusters.

When to use it:
- Reporting in Power BI or other BI tools
- Business analysts and power users who work mainly in SQL
- Interactive dashboards and ad-hoc queries on the Gold layer

Differences from an all-purpose cluster:
- Billing is based on DBU SQL (different rates).
- Provisioning and scaling are automatic.
- BI workloads are isolated from engineering clusters.



### SQL Warehouse: dedicated compute for SQL and BI

A SQL Warehouse is a separate compute type, decoupled from notebook clusters. It is designed for:
- analysts who use the SQL Editor
- BI tools: Tableau, Power BI, Looker
- AI/BI Dashboards and Genie in Databricks

SQL Warehouse types:

| Type | Startup | Cost | When to use |
|------|---------|------|------------|
| Serverless | Seconds | Higher DBU/hr, billed per second while running; auto-stops when idle | Interactive SQL work, variable workloads |
| Pro | 3-5 min | Medium, rich features | Ad-hoc analytics, advanced workloads |
| Classic | 3-5 min | Lowest, basic features | Simple SQL queries, CI/CD query testing |

How to create a SQL Warehouse:
1. Left menu → **SQL Warehouses**
2. Click **Create SQL Warehouse**.
3. Choose the type: **Serverless** (recommended for training) if your workspace offers it. Some workspaces, for example trials, show only **Pro** and **Classic**; then choose **Pro**.
4. Cluster Size: `X-Small` (enough for the demo; the dialog may suggest a larger size).
5. Auto Stop: `10 minutes` (saves cost).

How to connect Power BI or Tableau:
1. Click your SQL Warehouse → **Connection details** tab.
2. Copy **Server hostname** and **HTTP path**.
3. In Power BI: Get Data → Azure → Azure Databricks.
4. Paste the server hostname and HTTP path, then sign in with Microsoft Entra ID.

> A notebook cluster is for data engineers (Python, PySpark, Scala). A SQL Warehouse is for SQL analysts and BI tools.
> You **can** attach a notebook to a SQL warehouse, but then only SQL (and Markdown) cells run. Python and PySpark cells fail and need a cluster or serverless compute.


## Access Modes (classic compute)

Every classic all-purpose or job compute has an access mode. It decides who can attach to it and which features work. Serverless needs no access mode.

| Access mode | Who can use it | Languages | Not supported |
|---|---|---|---|
| Standard (formerly *shared*) | Many users at once, with isolation between them | Python, SQL, Scala | R, RDD API, Databricks Runtime for ML, GPU, spark-submit tasks |
| Dedicated (formerly *single user*) | One user, or one group (group access: Public Preview) | Python, SQL, Scala, R | None of the Standard limits (use it for R, RDD, ML runtime, GPU) |
| Auto (default in the UI) | Picks Standard. If you choose an ML runtime, a GPU instance type or a runtime below 14.3, it picks Dedicated | | |

> Run production jobs on serverless jobs or job compute, never on an all-purpose cluster. Databricks recommends Standard access mode unless the workload needs something that Standard does not support.


## When to Use Which Compute?

| Task | Compute | Why |
|---|---|---|
| Data exploration in a notebook | Serverless | No cluster to manage; all labs use it |
| Production ETL job | Job Cluster or serverless jobs | Ephemeral, isolated, cheaper |
| Lakeflow Spark Declarative Pipeline (formerly DLT) | Serverless (default) or classic pipeline compute | The pipeline provisions and manages its own compute |
| SQL query for analysis | SQL Warehouse | Optimised SQL, auto-scale |
| Power BI / Tableau dashboard | SQL Warehouse | JDBC/ODBC, certified connector |
| Ad-hoc analyst query | SQL Warehouse | Fast start, no Spark overhead |
| Deep learning training on GPU | AI Runtime (serverless GPU, Public Preview) or classic compute with ML runtime (Dedicated) | Standard access mode and CPU serverless have no GPU |

> A simple rule:
> - Writing code (Python / PySpark): serverless. Use a classic cluster only for what serverless does not support (R, RDD, Spark UI).
> - SQL analytics or BI: SQL Warehouse.

### Cluster Info

You need the runtime version and the Photon status when you compare performance or report a problem. The cell prints the Spark version, plus the runtime version and Photon status where the compute exposes them. Serverless does not expose the runtime setting, so it prints `n/a on serverless`. Photon is always on in serverless, which is why the query plan and the Query Profile later in this notebook show Photon operators. On a classic cluster you see the real values of both settings.

In [0]:
# Cluster runtime and Photon status
# Serverless does not expose spark.databricks.* cluster confs (reading one raises
# CONFIG_NOT_AVAILABLE), so each read falls back to a placeholder
def safe_conf(key, fallback="n/a on serverless"):
    try:
        return spark.conf.get(key)
    except Exception:
        return fallback

dbr_version = safe_conf("spark.databricks.clusterUsageTags.sparkVersion")
photon = safe_conf("spark.databricks.photon.enabled", "always on in serverless")
print(f"Spark version: {spark.version}  |  Runtime: {dbr_version}  |  Photon: {photon}")


On serverless, the Spark version is the only value read from the compute.

### Monitoring

On classic compute, look for problems in Cluster → Event log, in the Spark UI (Jobs and SQL tabs) and in the driver and executor logs.

On serverless there is no cluster page and no Spark UI. Use the **Query Profile** (**See performance** under the code, above the result) and **query history**.

> For production pipelines, log to Delta tables, not only to cluster logs.

## Spark Architecture


## 1. Driver & Executors: cluster architecture

When you run a cell in a notebook, Databricks does not execute it on your laptop. The code goes to a **Spark cluster**, which consists of a Driver and Executors:

<img src="../../../assets/images/5ea7056c932f4759afa86902f27317fe.png" width="860">

### Driver
- One process per cluster that coordinates all work
- Holds the `SparkSession`, builds the query plan and schedules tasks
- If the Driver crashes, the job crashes
- **Your Python code runs on the Driver**: loops, list comprehensions and `print()` all run there

### Executors
- Many processes on worker nodes that do the actual computation
- Each Executor processes one or more partitions in parallel
- Results return to the Driver (for example with `collect()`) or are written to storage

### Partition: unit of parallelism
Spark divides data into partitions (about 128 MB each). One partition is one task on one core of one Executor.

![Spark Partitions](../../../assets/images/9b9d5e3d62aa41119f758b11ea5a1c60.png)

> Spark does not move data to the computation. It sends small pieces of code to where the data is (data locality). That is why it scales to petabytes.


## 2. Lazy Evaluation: code that doesn't run immediately

Lazy evaluation is the most important concept in Spark, and it confuses many beginners.

### Transformations vs Actions

| Type | Examples | What happens |
|------|----------|-------------|
| Transformation (lazy) | `filter()`, `select()`, `groupBy()`, `join()`, `withColumn()`, `orderBy()` | Nothing runs. Spark records the plan but does not compute |
| Action (eager) | `count()`, `show()`, `collect()`, `write...save()` / `saveAsTable()`, `display()`, `first()` | Spark runs the whole plan and returns the result |

### Why is laziness good?
Spark sees the whole plan before execution, so it can:
- move a filter **before** an expensive join (predicate pushdown)
- remove columns you don't need (column pruning)
- merge multiple `filter()` calls into a single pass over the data
- choose the best join strategy (broadcast vs shuffle)

The next cells time each step on the `samples.nyctaxi.trips` table. Transformations return at once; the time goes into the action that reads the data.


In [0]:
# Lazy evaluation, step 1: spark.table() only registers the plan, it reads no data
import time

t0 = time.time()
df = spark.table("samples.nyctaxi.trips")
print(f"spark.table() took {time.time() - t0:.3f}s and returned a {type(df).__name__}, not data")


In [0]:
# Step 2: transformations only extend the plan, still nothing is computed
t0 = time.time()
df2 = (df
    .filter(F.col("trip_distance") > 2)
    .withColumn("fare_per_mile", F.round(F.col("fare_amount") / F.col("trip_distance"), 2))
    .select("pickup_zip", "trip_distance", "fare_amount", "fare_per_mile")
)
print(f"filter/withColumn/select took {time.time() - t0:.3f}s")


In [0]:
# Step 3: count() is an action, so Spark now reads the data and runs the whole plan
t0 = time.time()
row_count = df2.count()
print(f"count() took {time.time() - t0:.2f}s and returned {row_count:,} rows")


In [0]:
# Show the parsed, analyzed and optimized logical plans and the physical plan, without running them
df2.explain(mode="extended")


At the end of the `explain()` output, the physical plan shows what Spark will actually run.

## 3. DAG, Stages and Shuffles

When an action triggers execution, Spark builds a **DAG** (Directed Acyclic Graph), a graph of all the steps. It splits the DAG into **stages** and sends them to the Executors.

### When does a new Stage appear?

A stage boundary is a **shuffle**: the moment when data must be redistributed between Executors.

| Operation | Shuffle? | Explanation |
|-----------|----------|-------------|
| `filter()`, `select()`, `withColumn()` | No (narrow) | Each partition is independent |
| `groupBy().agg()` | Yes (wide) | Data by key must reach the same Executor |
| `join()` (large tables) | Yes (wide) | Matching keys must be co-located |
| `orderBy()` / `sort()` | Yes (wide) | Global sort requires collecting data |
| `repartition(n)` | Yes | Explicit redistribution |
| `coalesce(n)` | No | Reducing partitions without full shuffle |
| Broadcast join | No | Small table is sent to every Executor |

> A shuffle is expensive because Spark writes the data to disk and sends it over the network. Fewer shuffles is the main Spark optimization technique.

<img src="../../../assets/images/nb01-code-to-stages.png" alt="From code to stages on samples.nyctaxi.trips: lazy filter and select, a wide groupBy, and the action that runs the whole plan as a DAG of stages" width="800">

## 4. Catalyst Optimizer + Adaptive Query Execution (AQE)

> **Self-study:** skipped in class, not required by any lab. AQE comes back in module 06.

### Catalyst Optimizer (compile-time)
Before Spark sends anything to Executors, Catalyst rewrites the logical plan into an optimal physical plan:

| Optimization | Example |
|--------------|---------|
| Predicate Pushdown | `df.join(other).filter(col > 10)` → the filter is placed before the join |
| Column Pruning | `df.select("a","b").filter(...)` → Spark doesn't read other columns from Parquet |
| Constant Folding | `filter(col > 100 - 50)` → Spark computes `50` at compile-time |
| Join Reordering | The cost-based optimizer picks the join order from table statistics |

### Adaptive Query Execution (AQE, runtime)
AQE works **during** execution. It improves the plan based on actual statistics:

| AQE Feature | What it does |
|-------------|-------------|
| Dynamic shuffle partitions | Reduces 200 → X partitions based on actual data size |
| Broadcast join conversion | If a table turns out to be small after a filter, AQE converts the join to a broadcast join |
| Skew handling | Detects and splits uneven (skewed) partitions |

> Because of Catalyst and AQE, **the same SQL and PySpark query generates an identical plan**. There is no performance difference between `spark.sql("SELECT...")` and the DataFrame API.

Check these settings on your compute before you tune shuffle partitions by hand.


In [0]:
# AQE and join settings (AQE is on by default since Spark 3.2)
# Serverless manages most of them, so only spark.sql.shuffle.partitions has a value there
for key in [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.shuffle.partitions",
]:
    print(f"{key:<50} {safe_conf(key, 'managed by Databricks on serverless')}")


On serverless, Databricks manages AQE and the join settings, so only `spark.sql.shuffle.partitions` shows a value. On classic compute all four keys show their values.

## 5. Spark UI: where to find execution information

The Spark UI is a built-in diagnostic interface that you open **directly from the cluster in Databricks** (classic compute).

> **Note:** On serverless there is no Spark UI under Compute. Use the **Query Profile** (**See performance** under the code, above the result) and **query history** instead.

### How to open Spark UI
1. Click **Compute** in the left menu.
2. Select your cluster.
3. Click **Spark UI** (tab at the top).

### Key Spark UI tabs

| Tab | What it shows | When to use |
|-----|---------------|-------------|
| Jobs | List of jobs: time, status, number of Stages | First stop when debugging |
| Stages | Details of each Stage: tasks, shuffle read/write, time | When a Stage is slow |
| SQL / DataFrame | Query plan with timing for each node | Optimizing SQL queries |
| Storage | Cached DataFrames (size, fraction cached) | Cache management |
| Executors | Memory and CPU usage per Executor | OOM debugging |

### What to look for in Spark UI when a job is slow
1. A long Stage: check shuffle read/write. A large shuffle is the bottleneck.
2. Uneven task times: data skew (some partitions are much larger).
3. High GC time: the Executor is out of memory. Increase executor memory or reduce the data.
4. Few tasks: few partitions, so the data is not processed in parallel.

### Key metrics to monitor in the Stages tab

- Task duration histogram: a uniform histogram is healthy, a long tail means skew.
- Shuffle Read/Write: large values point to expensive wide transformations.
- Spill (Memory) / Spill (Disk): non-zero values point to memory pressure.
- Input size vs. output size: a very large output compared with the input may point to a fan-out problem.

The next query groups and sorts the taxi trips. On classic compute the Spark UI shows the shuffle between its stages. On serverless the Query Profile shows the scan, aggregate, filter and sort operators with their rows and time; a separate shuffle (Exchange) node may not appear in its default graph.


In [0]:
# After the run: serverless → See performance (under the code, above the result) → Query Profile;
# classic → Spark UI → Jobs → last job → Stage (tasks, duration, shuffle read/write)
result = (spark.table("samples.nyctaxi.trips")
    .groupBy("pickup_zip", "dropoff_zip")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
    )
    .filter(F.col("trips") >= 10)
    .orderBy(F.desc("total_revenue"))
)
display(result.limit(15))

The result shows the 15 pickup and dropoff zip pairs with at least 10 trips and the highest total revenue.

## Summary

### What was achieved
- You learned the Lakehouse concept as an evolution of the data lake and the data warehouse.
- You explored the platform elements: Workspace, Catalog, Compute.
- You saw the Unity Catalog hierarchy: Metastore → Catalog → Schema → Objects.
- You practiced magic commands: `%sql` and `%run` (reference: `%python`, `%md`, `%fs`, `%sh`, `%pip`).
- You compared Hive Metastore and Unity Catalog.
- You created your first Delta table in Unity Catalog with the 3-level namespace.

### Key Takeaways
1. The Lakehouse removes data duplication: a single copy serves BI, ML and real-time analytics.
2. Unity Catalog is the foundation of governance: 3-level namespace, fine-grained permissions, automatic lineage.
3. Clusters are flexible. Autoscaling and spot instances reduce costs, and Photon speeds up queries.
4. Notebooks mix languages: SQL and Python, magic commands, Git integration through Git Folders.
5. Delta Lake is the default format: ACID transactions, time travel, schema evolution.

### Quick Reference: Key Commands

| Operation | PySpark | SQL |
|-----------|---------|-----|
| Set catalog | `spark.sql(f"USE CATALOG {CATALOG}")` | `USE CATALOG my_catalog` |
| List catalogs | `spark.sql("SHOW CATALOGS")` | `SHOW CATALOGS` |
| List schemas | `spark.sql("SHOW SCHEMAS")` | `SHOW SCHEMAS` |
| Create table | `df.write.saveAsTable("cat.schema.table")` | `CREATE TABLE cat.schema.table AS SELECT ...` |
| Read table | `spark.table("cat.schema.table")` | `SELECT * FROM cat.schema.table` |
| Metadata | - | `SELECT * FROM system.information_schema.tables` |
| Install lib | `%pip install package` | - |

**<a href="$../../..">Course folder (README)</a>** | <a href="$./02_data_ingestion">02 — Data Ingestion →</a>

## Self-check (trainer): participants can skip this cell

In [ ]:
# Trainer self-check: facts the demo shows, asserted on the state left at the end
assert note == NOTE_TEXT, f"Volume file read back {note!r}, expected {NOTE_TEXT!r}"
assert spark.table(table_name).count() == len(sales_data), f"{table_name} should have {len(sales_data)} rows"
assert row_count > 0, "the lazy evaluation filter returned no taxi trips"
print("Self-check passed")